In [ ]:
#I heavily recommend running this code in a Jupyter Notebook or similar environment to make package management easier
#This script needs over 50 packages to run, so it is not recommended to run it in a standard Python environment without proper package management.
#
#This script loads customer provided Excel data and shapefile data from Digiroad.
#These files are merged based on the road number and segment number.
#Condition is matched based on the AET and LET values from the shapefile and the Aet and Let values from the Excel file.
#After merging, the script creates a new column "kunto_luokka" based on the "RMS_yhd" values, which represent the road condition.
#Finally, the merged dataset is saved to a GeoPackage file named "road_condition.gpkg", which can be used in QGIS for visualization.
import geopandas as gpd
from tqdm import tqdm
import pandas as pd
from pathlib import Path

#------Create an output folder if it doesn't exist to save the results.------

#This gets the folder where the script is located, which is assumed to be the "codes" folder in the project structure.
#Made this to work with both Jupyter Notebook and standard Python environments, as __file__ is not defined in Jupyter Notebooks.
try:
    codes_dir = Path(__file__).resolve().parent
except NameError:
    codes_dir = Path.cwd()

#project root directory (one level up from the codes directory)
project_root = codes_dir.parent

#------Create an output folder if it doesn't exist to save the results.------
output_folder = project_root / "output"
output_folder.mkdir(exist_ok=True)

data_folder = project_root / "Data"

print(data_folder)
print(output_folder)

#----------------------------------------------------
# Load all excel files from the Data folder
#----------------------------------------------------
excel_files = list(data_folder.glob("*.xlsx"))

dfs = []
for file in tqdm(excel_files, desc="Loading Excel files"):
    excel = pd.read_excel(file, header=0)
    print(f"Loaded {file.name} with {len(excel)} rows.")
    excel.columns = excel.columns.str.strip().str.lower()

    excel["source_file"] = file.name  # Add a column to identify the source file
    dfs.append(excel)

excel = pd.concat(dfs, ignore_index=True)
# excel = excel.drop_duplicates()

print("Total rows in excel: ", len(excel))

#----------------------------------------------------
# Load Digiroad geopackage data (KokoSuomi_Digiroad_R_GeoPackage.gpkg)
#----------------------------------------------------
roads = gpd.read_file(
    r"C:\School\TieDataProject\KokoSuomi_Digiroad_R_GeoPackage\KokoSuomi_Digiroad_R_GeoPackage.gpkg",
    layer="DR_LINKKI",
    columns=["TIENUMERO", "TIEOSANRO", "AET", "LET", "geometry"]
)
print("Total rows in roads: ", len(roads))

# print("road ORIGINAL", roads.columns.tolist())
# print("excel ORIGINAL", excel.columns.tolist())

#---------We rename the columns in the both datasets to match for merging.---------
excel.columns = excel.columns.str.strip().str.lower()

excel = excel.rename(columns={
    "tieosanumero": "aosa",
    "aet": "excel_aet",
    "let": "excel_let",
})

#-----From the roads dataset, we select the relevant columns.-------
roads.columns = roads.columns.str.strip().str.lower()
roads = roads[["tienumero","tieosanro","aet","let","geometry"]]

#---------We rename the columns in the roads dataset to match the excel dataset for merging.---------
roads = roads.rename(columns={
    "tienumero": "tie",
    "tieosanro": "aosa",
    "aet": "road_aet",
    "let": "road_let",
})

roads = roads[roads["tie"].isin(excel["tie"])] # Filter roads to only include those that are present in the excel dataset.

print("Unique roads in Excel:", excel["tie"].nunique())
print("Unique roads in roads:", roads["tie"].nunique())

print("road MODDED", roads.columns.tolist())
print("excel MODDED", excel.columns.tolist())

In [ ]:
#--------Merging the two datasets on the "Tie" and "Aosa" columns, which represent the road number and segment number.---------
merged = roads.merge(excel, on=["tie", "aosa"], how="left")

if "ajorata" not in merged.columns:
    merged["ajorata"] = 1

if "kaista" not in merged.columns:
    merged["kaista"] = 1

merged["ajorata"] = merged["ajorata"].fillna(1)
merged["kaista"] = merged["kaista"].fillna(1)

# Overlap filter with tolerance
EPS = 0.01

merged = merged[
    (
        (merged["excel_aet"] <= merged["road_let"] + EPS) &
        (merged["excel_let"] >= merged["road_aet"] - EPS)
    )
    |
    (merged["excel_aet"].isna())
].copy()

print("merged", merged.columns.tolist())

#----------------------------------------------------
# Lane separation and offset calculation
#----------------------------------------------------
direction_gap = 2
lane_width = 2.5

merged["kaista"] = merged["kaista"].fillna(11)

def separate_kaista(kaista):
    kaista = int(kaista)

    direction = kaista // 10     # 11 → 1, 21 → 2
    lane_num = kaista % 10       # 11 → 1, 23 → 3

    return direction, lane_num

def compute_offset(row):
    ajorata = int(row["ajorata"])
    kaista = int(row["kaista"])

    # direction: 1 = right side, 2 = left side
    direction = 1 if ajorata == 1 else -1

    # distance from centerline
    lane_offset = (kaista - 0.5) * lane_width

    # add gap between directions
    total_offset = direction * (direction_gap + lane_offset)

    return total_offset


tqdm.pandas(desc="Creating lane geometries")
merged["offset"] = merged.progress_apply(compute_offset, axis=1)

def safe_offset(row):
    try:
        return row.geometry.parallel_offset(
            row["offset"],
            side="left",
            join_style=2
        )
    except Exception:
        return row.geometry  # fallback if it fails

merged["geometry"] = merged.progress_apply(safe_offset, axis=1)

merged = merged.explode(index_parts=False)

#------Droping the Aet_excel and Let_excel columns as they are no longer needed after filtering.----------
#merged = merged.drop(columns=["excel_aet", "excel_let"])

#-------Printing the shape and head of the merged dataset to verify the results before plotting and saving to file.---------
print(merged.shape)
print(merged.head())

#Saving the merged dataset to a GeoPackage file named "road_condition.gpkg".
#This is the file that will be used in the QGIS project to visualize the road conditions.
merged.to_file(output_folder / "road_condition.gpkg", driver="GPKG")
